In [1]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import EsmForMaskedLM, PretrainedConfig, EsmTokenizer

from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops
import yaml
import sys
import json
import functools

import numpy as np
from huggingface_hub import hf_hub_download

/home/averma2/miniforge3/envs/exploratory_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm

In [3]:
device = "cuda"
CONTEXT_LEN = 1000

In [125]:
# ESM-2 config
with open("./esm_config150M.json", "r") as file:
    esm_config = json.load(file)
model_name = esm_config["_name_or_path"]
model_name = model_name[model_name.find("facebook"):]
esm_config["token_dropout"] = False
esm_config["model_name"] = model_name
esm_config = PretrainedConfig(**esm_config)

# ESM-2 tokenizezr config
REPO_ID = esm_config.model_name
vocab_file = "vocab.txt"
special_tokens_map_file = "special_tokens_map.json"
tokenizer_config = {}
tokenizer_config["vocab_file"] = hf_hub_download(repo_id=REPO_ID, filename=vocab_file)
tokenizer_config["model_max_length"] = CONTEXT_LEN
with open(hf_hub_download(repo_id=REPO_ID, filename=special_tokens_map_file), "r") as f:
    tokenizer_config = {**tokenizer_config, **(json.load(f))}

# hooked ESM-2 config
hooked_esm_config = get_hooked_esm_config(esm_config, context_len=CONTEXT_LEN)

In [126]:
# ESM-2 35M
model_name = esm_config.model_name
tokenizer = tokenizer = EsmTokenizer(**tokenizer_config)
reg_esm = EsmForMaskedLM(esm_config).to(device)

# hooked ESM-2
hooked_esm = HookedTransformer(hooked_esm_config)
hooked_esm.load_state_dict(get_hooked_state_dict(reg_esm.state_dict(), hooked_esm_config))

# helper function to get logits from hooked ESM-2 head
apply_esm_lm_head = functools.partial(get_logits_hooked_esm, ESM2_lm_head=reg_esm.lm_head)
print(model_name)

facebook/esm2_t30_150M_UR50D


<img src="./ESM2_architecture.png" width="1000"/>

In [156]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
    
pathogens = list(config["pathogens"].keys())
pathogen_idx = 7
print(pathogens[pathogen_idx])
fasta_file = f"../../data/pathogen/{pathogens[pathogen_idx]}/alignment.fasta"
data = load_sequences(fasta_file)
sequence_names, sequences = list(zip(*list(data.items())))

influenza_h1n1pdm_ha


In [157]:
print(tokenizer.model_max_length)

1000


In [158]:
N = 0
test_seq = np.unique(sequences)[50]
# test_seq = np.unique(sequences)[1000]
print(test_seq)
tokenized_seq = tokenizer(test_seq, return_tensors="pt", padding=False).input_ids[:,:hooked_esm_config.n_ctx].to(device)
print(tokenized_seq.shape)

DTLCIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDKHDGKLCKLRGVAPLHLGQCNIAGWILGNPECESLSTASSWSYIVETPSSDNGTCYPGDFIDYEELREQLSSVSSFERFEIFPKASSWPNHDSDNGVTAACPHAGAKSFYKNLIWLVKKGKSYPKISKSYINDQGKEVLVLWGIHHPSTITDQESLYQNADAYVFVGSSRYSKKFKPEIAIRPKVRDRAGRMNYYWTLVEPGDKITFEATGNLVAPRYAFAMKKDAGSGIIISDTPVHDCNTTCQTPKGAINTSLPFQNIHPITIGKCPKYVRSTKLRLATGLRNIPSIQSR
torch.Size([1, 329])


In [159]:
print(tokenized_seq)

tensor([[ 0, 13, 11,  4, 23, 12,  6, 19, 21,  5, 17, 17,  8, 11, 13, 11,  7, 13,
         11,  7,  4,  9, 15, 17,  7, 11,  7, 11, 21,  8,  7, 17,  4,  4,  9, 13,
         15, 21, 13,  6, 15,  4, 23, 15,  4, 10,  6,  7,  5, 14,  4, 21,  4,  6,
         16, 23, 17, 12,  5,  6, 22, 12,  4,  6, 17, 14,  9, 23,  9,  8,  4,  8,
         11,  5,  8,  8, 22,  8, 19, 12,  7,  9, 11, 14,  8,  8, 13, 17,  6, 11,
         23, 19, 14,  6, 13, 18, 12, 13, 19,  9,  9,  4, 10,  9, 16,  4,  8,  8,
          7,  8,  8, 18,  9, 10, 18,  9, 12, 18, 14, 15,  5,  8,  8, 22, 14, 17,
         21, 13,  8, 13, 17,  6,  7, 11,  5,  5, 23, 14, 21,  5,  6,  5, 15,  8,
         18, 19, 15, 17,  4, 12, 22,  4,  7, 15, 15,  6, 15,  8, 19, 14, 15, 12,
          8, 15,  8, 19, 12, 17, 13, 16,  6, 15,  9,  7,  4,  7,  4, 22,  6, 12,
         21, 21, 14,  8, 11, 12, 11, 13, 16,  9,  8,  4, 19, 16, 17,  5, 13,  5,
         19,  7, 18,  7,  6,  8,  8, 10, 19,  8, 15, 15, 18, 15, 14,  9, 12,  5,
         12, 10, 14, 15,  7,

In [160]:
# run respective models
with torch.no_grad():
    outputs = reg_esm.forward(tokenized_seq, output_hidden_states=True)
    reg_esm_hs = outputs.hidden_states

hooked_toks, cache = hooked_esm.run_with_cache(tokenized_seq)

In [161]:
for l in range(esm_config.num_hidden_layers + 2):
    # 0th layer is embedding layer
    if l < esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"blocks.{l}.hook_resid_pre"]
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"ln_final.hook_normalized"]

    # ESM LM head:
    else:
        actual_output = outputs.logits
        with torch.no_grad():
            hooked_output = apply_esm_lm_head(hooked_toks)
    
    is_close = torch.all(torch.isclose(actual_output, hooked_output, rtol=0.1)).item()
    print(f"layer {l}: {is_close}")

layer 0: True
layer 1: False
layer 2: True
layer 3: True
layer 4: False
layer 5: False
layer 6: False
layer 7: True
layer 8: False
layer 9: False
layer 10: False
layer 11: False
layer 12: False
layer 13: False
layer 14: False
layer 15: False
layer 16: False
layer 17: False
layer 18: False
layer 19: False
layer 20: False
layer 21: True
layer 22: False
layer 23: False
layer 24: False
layer 25: True
layer 26: False
layer 27: False
layer 28: False
layer 29: False
layer 30: True
layer 31: True


In [162]:
print(outputs.logits[0])

tensor([[-0.1851,  0.0000,  0.7524,  ...,  0.3136,  0.0356, -1.1224],
        [-0.2950,  0.0000,  0.6416,  ...,  0.1122,  0.3024, -1.2474],
        [-0.2563,  0.0000,  0.8130,  ...,  0.2925,  0.0921, -0.8738],
        ...,
        [-0.1879,  0.0000,  0.7180,  ...,  0.3027,  0.1340, -1.1337],
        [-0.2945,  0.0000,  0.7246,  ...,  0.2832,  0.1495, -1.1273],
        [-0.2956,  0.0000,  0.8153,  ...,  0.2576,  0.0035, -1.1730]],
       device='cuda:0')


In [163]:
l1 = torch.argmax(outputs.logits[0], dim=-1)
l2 = torch.argmax(apply_esm_lm_head(hooked_toks)[0], dim=-1)

print(torch.equal(l1,l2))
print(l1, "\n")
print(l2)

True
tensor([21, 21, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
         2, 21, 21, 24, 21, 21, 21,  2, 21,  2, 21, 21, 21, 21, 21, 21, 24, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
         2, 21, 21, 16, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 16, 21, 24, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 16, 21, 21, 24, 21, 21, 21, 21,  2, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 16,
        21, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 16,
        21, 21, 21, 21, 21, 16, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 16, 21,
        16, 21, 21, 21, 21, 21, 21,